# Ti-6Al-4V SLM/LPBF Vickers Hardness — 논문 방법론 적용 예비 모델

**참고 논문:** *Predictability assessment of as-built hardness of Ti-6Al-4V alloy fabricated via laser powder bed fusion*  
DOI: 10.1016/j.mfglet.2023.08.113

**연구 목적:** 자체 수집 소재 DB(`데이터베이스_소재DB.xlsx`)의 SLM/LPBF Ti-6Al-4V 시편에 대해, 논문의 공정조건 기반 예측 방법론을 재현 가능한 범위 내에서 적용하여 Vickers hardness 회귀 성능을 비교·평가한다.

**중요한 원칙:**
- 데이터 누수 금지 — 모든 scaling/feature engineering 은 Pipeline 안에서 수행
- test set 은 튜닝에 사용하지 않음
- target column 은 입력 피처에 절대 포함하지 않음
- R² 가 비정상적으로 높으면 누수 의심하고 점검
- 데이터 수가 적을 수 있어 5-Fold CV 와 LOOCV 를 모두 비교

## 0. 환경 / 라이브러리 / 한글 폰트

In [ ]:
import os, warnings, importlib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, LeaveOneOut, cross_val_score, cross_val_predict
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
np.random.seed(42)

# 한글 폰트 (Windows)
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# 선택적 패키지 — XGBoost / LightGBM 설치 여부 자동 감지
HAS_XGB = importlib.util.find_spec('xgboost') is not None
HAS_LGBM = importlib.util.find_spec('lightgbm') is not None
if HAS_XGB:
    from xgboost import XGBRegressor
if HAS_LGBM:
    from lightgbm import LGBMRegressor
print(f'XGBoost installed:  {HAS_XGB}')
print(f'LightGBM installed: {HAS_LGBM}')

## 1. 엑셀 파일 구조 확인

`데이터베이스_소재DB.xlsx` 는 **헤더가 12행(0-index 11)**, 13행이 단위(`µm, W, mm/s, ...`), 14행부터 실제 데이터인 비정형 시트다. 따라서 `header=11` 로 읽고 첫 행(단위)을 제거한 뒤 사용한다.

In [ ]:
XLSX = 'data/데이터베이스_소재DB.xlsx'
xl = pd.ExcelFile(XLSX)
print('=== 시트 목록 ===')
for s in xl.sheet_names:
    print(' -', s)

In [ ]:
# 경도 시트: header=11 (= 항목 행), 첫 번째 행은 단위(µm, W, mm/s, ...) → 제거
raw = pd.read_excel(XLSX, sheet_name='경도', header=11)
print('--- 원본 컬럼 ---')
for c in raw.columns:
    print(f'  {c!r:30}')
print(f'\n원본 shape: {raw.shape}')
print('\n첫 번째 행 (단위 row):')
print(raw.iloc[0])

In [ ]:
# 단위 row 제거 + 컬럼명 정리 (공백 strip)
df = raw.iloc[1:].reset_index(drop=True).copy()
df.columns = [str(c).strip() for c in df.columns]

# 의미 있는 컬럼만 선택 (Unnamed 제거)
keep_cols = [c for c in df.columns if not c.startswith('Unnamed') and c != '항목']
df = df[keep_cols]

print('=== 정리된 컬럼 ===')
for c in df.columns:
    print(f'  {c!r:25}  dtype = {df[c].dtype}')
print(f'\nshape: {df.shape}')
print(f'\n=== 결측치 개수 ===')
print(df.isna().sum())

In [ ]:
# 숫자형 변환 (errors='coerce' 로 문자열 등은 NaN 처리)
numeric_candidates = ['층 두께', '레이저 파워', '스캔 속도', '에너지 밀도', '비커스', '로크웰',
                      '평균압흔크기1', '평균압흔크기2']
for c in numeric_candidates:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Target 후보 자동 탐색
target_candidates = []
for c in df.columns:
    cl = str(c).lower().replace(' ', '')
    if any(k in cl for k in ['비커스', 'vickers', 'hv', '경도', 'hardness']):
        if df[c].dtype.kind in 'fi' and df[c].notna().sum() > 0:
            target_candidates.append((c, int(df[c].notna().sum()), df[c].dtype))

print('=== Target column 후보 (비커스/HV/경도 키워드 + 숫자형) ===')
for name, n_valid, dt in target_candidates:
    print(f'  {name!r:20}  유효값 = {n_valid:3d}개   dtype = {dt}')

In [ ]:
# Target 결정: 비커스 (HV) — 가장 유효값 많고 연속형
TARGET = '비커스'
BASE_FEATURES = ['층 두께', '레이저 파워', '스캔 속도', '에너지 밀도']

# 분석용 데이터프레임: BASE + TARGET 모두 유효한 행
data = df[BASE_FEATURES + [TARGET]].copy()
print('Target 결측 제거 전 shape:', data.shape)
data = data.dropna(subset=[TARGET]).reset_index(drop=True)
print('Target 결측 제거 후 shape:', data.shape)

print('\n=== 입력 변수 결측 (target 보유 행 한정) ===')
print(data[BASE_FEATURES].isna().sum())

print('\n=== 기술통계 ===')
display(data.describe().round(3))

print('\n=== 중복 행 검사 (삭제하지 않고 보고만) ===')
n_dup = data.duplicated().sum()
print(f'  중복 행: {n_dup}')
if n_dup > 0:
    print('  주의: 동일 공정조건이 반복되는 것은 측정 반복일 수 있으므로 임의 삭제하지 않음.')

## 2. 입력/타겟 분포 빠른 점검

In [ ]:
fig, axes = plt.subplots(1, len(BASE_FEATURES) + 1, figsize=(20, 3.5))
for ax, c in zip(axes, BASE_FEATURES + [TARGET]):
    ax.hist(data[c], bins=20, edgecolor='k', alpha=0.85)
    ax.set_title(c); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('=== Pearson 상관 (vs 비커스) ===')
print(data.corr(numeric_only=True)[TARGET].round(3))

## 3. Feature Engineering — 4가지 피처 셋 정의

논문 (Sciencedirect 2213846323001748) 은 power, speed, layer thickness, hatch spacing 같은 1차 공정변수 외에 **에너지 밀도와 그 조합 변형**을 함께 비교한다. 본 연구에서는 hatch spacing 정보가 없으므로 다음 4셋으로 구성한다.

| 셋 | 구성 |
|---|---|
| **FS-A: Base 4** | 층 두께, 레이저 파워, 스캔 속도, 에너지 밀도 |
| **FS-B: Base + Ratios** | FS-A + (P/V), (P·t), (V·t), (P/(V·t)) |
| **FS-C: Base + log1p** | FS-A + log1p(P/V), log1p(P·t), log1p(V·t), log1p(P/(V·t)) |
| **FS-D: Polynomial(2차)** | FS-A 의 2차 다항식 (interactions + 제곱항) — Pipeline 내에서 적용 |

**다중공선성 주의:** 에너지 밀도 ≈ P / (V · t · h) 로 정의되므로 (h: hatch spacing 미보유), 본 데이터에서 `P/(V·t)` 는 에너지 밀도 × hatch spacing 의 상수배 형태일 가능성이 매우 높다. 이 경우 `에너지 밀도` 와 `P/(V·t)` 는 **수치적으로 거의 선형 종속**이다. 선형 모델(Linear / Ridge / Lasso / ElasticNet)에서는 회귀계수 분산이 커지고 해석이 불안정해질 수 있고, 트리/커널 모델에서는 영향이 덜하지만 중복 정보로 특별한 이득은 없다. 아래 셀에서 실제 상관/VIF 를 점검한다.

In [ ]:
def add_derived(df_in, log_version=False):
    """P/V, P·t, V·t, P/(V·t) 계산. log_version=True 면 log1p 변환 적용."""
    P = df_in['레이저 파워']
    V = df_in['스캔 속도']
    t = df_in['층 두께']
    out = df_in.copy()
    pairs = {
        'P/V':       P / V,
        'P·t':       P * t,
        'V·t':       V * t,
        'P/(V·t)':   P / (V * t),
    }
    for k, v in pairs.items():
        if log_version:
            out[f'log1p({k})'] = np.log1p(v)
        else:
            out[k] = v
    return out

df_FS_A = data[BASE_FEATURES].copy()
df_FS_B = add_derived(data[BASE_FEATURES], log_version=False)
df_FS_C = add_derived(data[BASE_FEATURES], log_version=True)
# FS-D 는 Pipeline 안에서 PolynomialFeatures(2)로 처리 — 누설 방지

print('--- FS-A (Base 4) ---'); print(df_FS_A.columns.tolist())
print('--- FS-B (+ ratios) ---'); print(df_FS_B.columns.tolist())
print('--- FS-C (+ log1p ratios) ---'); print(df_FS_C.columns.tolist())

print('\n=== 다중공선성 점검: 에너지 밀도 vs P/(V·t) ===')
corr_dup = np.corrcoef(df_FS_B['에너지 밀도'], df_FS_B['P/(V·t)'])[0, 1]
print(f'corr(에너지 밀도, P/(V·t)) = {corr_dup:+.6f}')
if abs(corr_dup) > 0.99:
    print('→ 거의 완전 선형 종속. 선형 모델은 둘 중 하나를 자동으로 무시하거나 계수가 폭발할 수 있음.')
    print('  본 분석에서는 두 변수를 동시 포함한 채로 Ridge/Lasso 의 정규화로 안정화한다.')

## 4. Pipeline / 모델 정의

**전처리 원칙 (누설 방지):**
- `SimpleImputer(strategy='median')` → median 대체
- 커널/선형 모델은 `StandardScaler` 적용 (Pipeline 내부)
- 트리 기반 모델은 scaling 불필요하지만 일관성 위해 동일 Pipeline 구조 (StandardScaler 는 트리에 영향 없음 → 그대로 적용해도 무해, 단 본 분석에서는 트리 모델은 passthrough 로 둠)
- FS-D(polynomial)는 `PolynomialFeatures(2, interaction_only=False)` 를 Pipeline 첫 단계에 끼워 fold-별로 fit
- 모든 fit 은 `train_test_split` 의 train fold 안에서만 수행

In [ ]:
def make_pipeline_for(model, kind='kernel', poly=False):
    """kind: 'kernel'(scaler 필요) | 'tree'(scaler 불필요) — Imputer 는 공통.
    poly=True 면 PolynomialFeatures(2) 를 imputer 다음에 추가."""
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if poly:
        steps.append(('poly', PolynomialFeatures(degree=2, include_bias=False)))
    if kind == 'kernel':
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)

# 모델 정의 — kind 정보로 Pipeline 구성 결정
MODEL_DEFS = {
    'LinearRegression':    (LinearRegression(),                 'kernel', {}),
    'Ridge':               (Ridge(random_state=42),             'kernel',
                            {'model__alpha': [1e-3, 1e-2, 1e-1, 1, 10, 100, 1000]}),
    'Lasso':               (Lasso(random_state=42, max_iter=20000), 'kernel',
                            {'model__alpha': [1e-4, 1e-3, 1e-2, 1e-1, 1, 10]}),
    'ElasticNet':          (ElasticNet(random_state=42, max_iter=20000), 'kernel',
                            {'model__alpha': [1e-3, 1e-2, 1e-1, 1, 10],
                             'model__l1_ratio': [0.1, 0.5, 0.9]}),
    'SVR':                 (SVR(kernel='rbf'),                  'kernel',
                            {'model__C':       [1, 10, 50, 100, 300, 500, 1000],
                             'model__gamma':   ['scale', 'auto', 0.001, 0.01, 0.1, 1],
                             'model__epsilon': [0.01, 0.05, 0.1, 0.2, 0.5]}),
    'RandomForest':        (RandomForestRegressor(random_state=42, n_jobs=-1), 'tree',
                            {'model__n_estimators':     [100, 300, 500],
                             'model__max_depth':        [None, 3, 5, 7, 10],
                             'model__min_samples_leaf': [1, 2, 3, 5]}),
    'ExtraTrees':          (ExtraTreesRegressor(random_state=42, n_jobs=-1), 'tree',
                            {'model__n_estimators':     [100, 300, 500],
                             'model__max_depth':        [None, 3, 5, 7, 10],
                             'model__min_samples_leaf': [1, 2, 3, 5]}),
    'GradientBoosting':    (GradientBoostingRegressor(random_state=42), 'tree',
                            {'model__n_estimators':  [100, 300, 500],
                             'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
                             'model__max_depth':     [2, 3, 4]}),
    'GaussianProcess':     (GaussianProcessRegressor(
                                kernel=C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2)) + WhiteKernel(1e-2, (1e-5, 1e1)),
                                normalize_y=True, alpha=1e-6, n_restarts_optimizer=5, random_state=42), 'kernel', {}),
}

# 선택적: XGBoost / LightGBM
if HAS_XGB:
    MODEL_DEFS['XGBoost'] = (XGBRegressor(random_state=42, n_jobs=-1, verbosity=0), 'tree',
                             {'model__n_estimators':  [100, 300, 500],
                              'model__learning_rate': [0.03, 0.05, 0.1],
                              'model__max_depth':     [3, 4, 5]})
if HAS_LGBM:
    MODEL_DEFS['LightGBM'] = (LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1), 'tree',
                              {'model__n_estimators':  [100, 300, 500],
                               'model__learning_rate': [0.03, 0.05, 0.1],
                               'model__num_leaves':    [15, 31, 63]})

print(f'사용 모델 수: {len(MODEL_DEFS)} → {list(MODEL_DEFS)}')

## 5. Train / Test Split

데이터 수가 적어 (`n=77`) test_size=0.2 와 0.25 둘 다 비교한다.  
튜닝은 **train fold 의 5-Fold CV** 로 수행 → test 는 최종 평가용으로만 사용 (튜닝에 미사용).

In [ ]:
y = data[TARGET].values
X_raw = data[BASE_FEATURES].values  # FS-A. FS-B/C 는 함수로 다시 만든다

for ts in [0.2, 0.25]:
    X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y, test_size=ts, random_state=42)
    print(f'test_size={ts:>4} → train={len(y_tr)}, test={len(y_te)}')

## 6. 메인 평가 루프

각 (모델, 피처셋) 조합에 대해:
1. train/test split (test_size=0.2, random_state=42)
2. GridSearchCV (5-Fold, scoring=r2) — train 만 사용
3. best estimator 로 train R², test R², test MAE, test RMSE
4. 전체 데이터 5-Fold CV (R² mean±std)
5. 전체 데이터 LOOCV (R², MAE)

Note: 데이터가 작아 GridSearchCV 의 cv split 안에서 또 작은 fold 로 나뉘므로 SVR 의 큰 그리드는 시간이 다소 걸린다. `n_jobs=-1` 로 병렬화.

In [ ]:
FEATURE_SETS = {
    'FS-A: Base 4':              df_FS_A.values,
    'FS-B: + ratios':            df_FS_B.values,
    'FS-C: + log1p ratios':      df_FS_C.values,
    'FS-D: poly(2) of Base 4':   df_FS_A.values,   # poly 는 Pipeline 단계에서 fit
}
FS_POLY = {'FS-D: poly(2) of Base 4': True}        # 표시용 플래그

TEST_SIZE   = 0.2
RANDOM_STATE = 42
CV_TUNE = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
CV_EVAL = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate_combo(model_name, model, kind, grid, X, y, poly=False):
    pipe = make_pipeline_for(model, kind=kind, poly=poly)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    if grid:
        gs = GridSearchCV(pipe, grid, cv=CV_TUNE, scoring='r2', n_jobs=-1, refit=True)
        gs.fit(X_tr, y_tr)
        best = gs.best_estimator_
        best_params = gs.best_params_
    else:
        best = pipe.fit(X_tr, y_tr)
        best_params = {}

    yhat_tr = best.predict(X_tr)
    yhat_te = best.predict(X_te)
    train_r2 = r2_score(y_tr, yhat_tr)
    test_r2  = r2_score(y_te, yhat_te)
    test_mae = mean_absolute_error(y_te, yhat_te)
    test_rmse = np.sqrt(mean_squared_error(y_te, yhat_te))

    # 5-Fold CV (전체 X, y) — 매 fold 마다 Pipeline 재학습 → 누설 없음
    fresh_pipe = make_pipeline_for(model.__class__(**model.get_params()), kind=kind, poly=poly)
    if grid:
        # best_params 적용
        fresh_pipe.set_params(**best_params)
    cv_r2 = cross_val_score(fresh_pipe, X, y, cv=CV_EVAL, scoring='r2', n_jobs=-1)
    cv_neg_mae = cross_val_score(fresh_pipe, X, y, cv=CV_EVAL, scoring='neg_mean_absolute_error', n_jobs=-1)

    # LOOCV — 동일 best 파이프
    loo_preds = cross_val_predict(fresh_pipe, X, y, cv=LeaveOneOut(), n_jobs=-1)
    loo_r2  = r2_score(y, loo_preds)
    loo_mae = mean_absolute_error(y, loo_preds)

    return {
        'train_R2':    train_r2,
        'test_R2':     test_r2,
        'test_MAE':    test_mae,
        'test_RMSE':   test_rmse,
        'CV5_R2_mean': float(np.mean(cv_r2)),
        'CV5_R2_std':  float(np.std(cv_r2)),
        'CV5_MAE_mean': float(-np.mean(cv_neg_mae)),
        'LOOCV_R2':    loo_r2,
        'LOOCV_MAE':   loo_mae,
        'best_params': best_params,
    }, best, (X_tr, X_te, y_tr, y_te)

In [ ]:
from time import time

results = []
best_estimators = {}
splits_cache = {}

for fs_name, X_fs in FEATURE_SETS.items():
    poly = FS_POLY.get(fs_name, False)
    print(f'\n=========== Feature set: {fs_name} (poly={poly}) ===========')
    for m_name, (model, kind, grid) in MODEL_DEFS.items():
        t0 = time()
        try:
            r, est, sp = evaluate_combo(m_name, model, kind, grid, X_fs, y, poly=poly)
            r['model'] = m_name; r['feature_set'] = fs_name
            results.append(r)
            best_estimators[(fs_name, m_name)] = est
            splits_cache[(fs_name, m_name)]    = sp
            dt = time() - t0
            print(f'  {m_name:18}  test_R²={r["test_R2"]:+.3f}  CV5={r["CV5_R2_mean"]:+.3f}±{r["CV5_R2_std"]:.3f}  '
                  f'LOOCV={r["LOOCV_R2"]:+.3f}  MAE={r["test_MAE"]:.2f}  ({dt:5.1f}s)')
        except Exception as e:
            print(f'  {m_name:18}  FAILED: {e}')

res_df = pd.DataFrame(results)
res_df = res_df[['feature_set','model','train_R2','test_R2','test_MAE','test_RMSE',
                 'CV5_R2_mean','CV5_R2_std','CV5_MAE_mean','LOOCV_R2','LOOCV_MAE','best_params']]
res_df = res_df.round({'train_R2':4,'test_R2':4,'test_MAE':3,'test_RMSE':3,
                       'CV5_R2_mean':4,'CV5_R2_std':4,'CV5_MAE_mean':3,'LOOCV_R2':4,'LOOCV_MAE':3})
os.makedirs('results', exist_ok=True)
res_df.to_csv('results/paper_method_full_comparison.csv', index=False, encoding='utf-8-sig')
print('\n저장: results/paper_method_full_comparison.csv')
res_df

## 7. 누수 / 과적합 자가 점검

- `train_R2 - test_R2` 가 너무 크면 (예: > 0.4) 과적합 가능성 높음
- `LOOCV_R2 ≫ test_R2` 또는 `train_R2 ≈ 1.000` 이면서 `test_R2` 가 낮으면 점검 필요
- `R²` 가 0.99+ 로 나오면 즉시 의심: 누수 / target 포함 / data leak in fold 점검

In [ ]:
diag = res_df[['feature_set','model','train_R2','test_R2','LOOCV_R2','CV5_R2_mean']].copy()
diag['gap_train_test'] = diag['train_R2'] - diag['test_R2']
diag = diag.sort_values('gap_train_test', ascending=False)
print('=== train-test gap 큰 순 ===')
display(diag.head(10).round(3))
if (diag['gap_train_test'] > 0.4).any():
    print('주의: 일부 모델은 train-test gap 이 0.4 초과 → 과적합 가능. 정규화 강화 또는 트리 깊이 축소 권장.')
if (res_df['test_R2'] > 0.99).any():
    print('경고: test R² > 0.99 인 조합 존재. target 포함/누수/중복행 점검 필요.')

## 8. 시각화 — 논문 Figure 5 스타일 비교

In [ ]:
# 8-1. 모델별 best (LOOCV R² 기준) 막대 비교 — 피처셋 색별 그룹
pivot_loo = res_df.pivot(index='model', columns='feature_set', values='LOOCV_R2')
pivot_mae = res_df.pivot(index='model', columns='feature_set', values='LOOCV_MAE')
fs_order = list(FEATURE_SETS.keys())
pivot_loo = pivot_loo[fs_order]
pivot_mae = pivot_mae[fs_order]

fig, axes = plt.subplots(2, 1, figsize=(13, 9))
pivot_loo.plot(kind='bar', ax=axes[0], edgecolor='black')
axes[0].set_title('모델 × 피처셋 — LOOCV R²', fontsize=13)
axes[0].set_ylabel('R²'); axes[0].axhline(0, color='k', lw=0.6)
axes[0].legend(title='Feature set', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].grid(axis='y', alpha=0.3)

pivot_mae.plot(kind='bar', ax=axes[1], edgecolor='black')
axes[1].set_title('모델 × 피처셋 — LOOCV MAE (HV)', fontsize=13)
axes[1].set_ylabel('MAE')
axes[1].legend(title='Feature set', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].grid(axis='y', alpha=0.3)
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.savefig('results/paper_method_R2_MAE.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8-2. Best 조합 식별 + Actual vs Predicted + Residual
best_idx = res_df['LOOCV_R2'].idxmax()
best_row = res_df.loc[best_idx]
best_fs, best_model = best_row['feature_set'], best_row['model']
print(f'BEST: {best_fs} / {best_model}  (LOOCV R² = {best_row["LOOCV_R2"]:.4f}, MAE = {best_row["LOOCV_MAE"]:.3f} HV)')

best_est = best_estimators[(best_fs, best_model)]
X_tr, X_te, y_tr, y_te = splits_cache[(best_fs, best_model)]
yhat_tr = best_est.predict(X_tr)
yhat_te = best_est.predict(X_te)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
ax = axes[0]
ax.scatter(y_tr, yhat_tr, alpha=0.6, label=f'Train (n={len(y_tr)})', edgecolor='k')
ax.scatter(y_te, yhat_te, alpha=0.85, label=f'Test (n={len(y_te)})', color='C3', edgecolor='k')
lim = [min(y.min(), yhat_tr.min(), yhat_te.min())-10, max(y.max(), yhat_tr.max(), yhat_te.max())+10]
ax.plot(lim, lim, 'k--', lw=1)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Actual hardness (HV)'); ax.set_ylabel('Predicted hardness (HV)')
ax.set_title(f'Actual vs Predicted\n{best_model} on {best_fs}')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
res_tr = y_tr - yhat_tr; res_te = y_te - yhat_te
ax.scatter(yhat_tr, res_tr, alpha=0.6, label='Train', edgecolor='k')
ax.scatter(yhat_te, res_te, alpha=0.85, label='Test', color='C3', edgecolor='k')
ax.axhline(0, color='k', lw=1)
ax.set_xlabel('Predicted hardness (HV)'); ax.set_ylabel('Residual (Actual − Predicted)')
ax.set_title('Residual plot'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/paper_method_best_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8-3. Feature importance (트리 기반 best가 트리이면) + Permutation importance (모델 무관)
from sklearn.inspection import permutation_importance

X_full = FEATURE_SETS[best_fs]
best_est.fit(X_full, y)  # 시각화용으로 전체 데이터에 다시 fit (해석용 — 평가에는 사용 안 함)

# 입력 피처 이름 결정
if FS_POLY.get(best_fs, False):
    poly_step = best_est.named_steps['poly']
    feat_names = poly_step.get_feature_names_out(BASE_FEATURES)
elif best_fs == 'FS-A: Base 4':
    feat_names = BASE_FEATURES
elif best_fs == 'FS-B: + ratios':
    feat_names = list(df_FS_B.columns)
elif best_fs == 'FS-C: + log1p ratios':
    feat_names = list(df_FS_C.columns)
else:
    feat_names = [f'f{i}' for i in range(X_full.shape[1])]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) tree feature_importances_
tree_model = best_est.named_steps['model']
if hasattr(tree_model, 'feature_importances_'):
    # poly 피처 변환 후 차원 일치
    fi = tree_model.feature_importances_
    if len(fi) == len(feat_names):
        imp = pd.Series(fi, index=feat_names).sort_values()
        imp.plot(kind='barh', ax=axes[0], color='C2', edgecolor='k')
        axes[0].set_title(f'{best_model} — feature_importances_')
    else:
        axes[0].text(0.5, 0.5, '피처 차원 불일치 — 생략', ha='center', va='center', transform=axes[0].transAxes)
else:
    axes[0].text(0.5, 0.5, f'{best_model} 은 feature_importances_ 미지원\n(아래 permutation 사용)',
                 ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_axis_off()

# (b) permutation importance (test set 위에서 계산 → 일반화 의미)
perm = permutation_importance(best_est, X_te, y_te, n_repeats=30, random_state=42, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=feat_names if len(perm.importances_mean)==len(feat_names) else [f'f{i}' for i in range(len(perm.importances_mean))]).sort_values()
perm_imp.plot(kind='barh', ax=axes[1], color='C0', edgecolor='k')
axes[1].set_title(f'Permutation importance (test set, {best_model})')

plt.tight_layout()
plt.savefig('results/paper_method_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n해석: permutation importance 가 0 근처면 그 변수는 일반화에 거의 기여하지 않음.')

## 9. 논문 vs 본 데이터 — 차이점 정리 (Markdown)

**Mfg. Letters 2023 (DOI 10.1016/j.mfglet.2023.08.113) 와 본 분석의 핵심 차이**

1. **데이터 출처와 분포**  
   - 논문: 여러 문헌에서 수집한 **literature-mined Ti-6Al-4V as-built hardness** 데이터셋. 다양한 장비/조성/장비 변종을 통합한 광범위한 분포.  
   - 본 분석: 단일 자체 수집 DB(`데이터베이스_소재DB.xlsx`). 시편 코드 `TIT*P*SS*` 형태의 **공정변수 grid** 위 측정. 분포 다양성과 표본 수가 논문보다 작음.  
   → 같은 모델/방법론이라도 R² 절대값이 더 낮을 수 있음. 

2. **표본 수와 과적합 위험**  
   - 본 데이터의 비커스 측정값은 약 77개. 모델 복잡도와 그리드 크기에 비해 작음.  
   - 논문에서 보고된 높은 R² 가 본 데이터에서 재현 안 되더라도 비정상이 아님. 오히려 **R² 가 너무 높게 나오면 누수 가능성 점검** 우선.

3. **공정변수 표현의 한계**  
   - 본 데이터는 hatch spacing 컬럼이 없어 에너지 밀도와 `P/(V·t)` 가 거의 선형 종속.  
   - **에너지 밀도 단일 변수만으로는 경도를 충분히 설명하기 어렵다.** Power 와 Speed 를 분리해 보는 표현이 더 풍부한 정보를 줌 (FS-A vs FS-B/C 비교 결과로 확인).

4. **혼재 변수 (confounders)**  
   - 시편 단위(SLM 빌드, 후열처리 여부, 측정 위치, Vickers 하중, 장비 변종) 차이는 본 DB 시트에서 다 분리되지 않음.  
   - 이런 혼재가 평균 경도에 잡음을 추가 → 모델이 같은 (P, V, t, E) 에 대해 두 다른 경도를 동시에 보면 R² 천정이 낮아짐.

5. **결론적 표현**  
   - 본 분석은 "논문 결과 재현"이 아니라 **"논문 방법론을 본 DB 에 적용한 예비 모델"** 이라고 표현해야 한다.  
   - 절대 성능이 낮아도 모델 간 상대비교, 피처셋 간 상대비교는 의미가 있다.

## 10. 최종 결론 (발표/보고서용 Markdown)

본 셀은 위 평가표의 best 조합을 자동으로 채워 결론 텍스트를 표시한다.

In [ ]:
# best 조합 정보 요약
br = res_df.loc[res_df['LOOCV_R2'].idxmax()]
print(f"""
## 최종 결론 (자동 요약)

### 연구 목적
자체 수집 SLM/LPBF Ti-6Al-4V 소재 DB 에 대해 공정변수 기반 Vickers hardness 회귀 모델을 구성하고,
참고 논문 (Mfg. Letters 2023, DOI 10.1016/j.mfglet.2023.08.113) 의 방법론을 재현 가능한 범위 내에서 적용·평가한다.

### 참고 논문 방법을 적용한 이유
- 논문은 Ti-6Al-4V LPBF as-built hardness 를 공정조건 기반 회귀로 다루는 직접 비교 대상
- 모델 패밀리(선형/커널/트리/GP), 피처 가공(에너지 밀도와 비율 변형), 검증 전략을 본 DB 규모에 맞춰 단순화하면 재현 가능

### 사용 변수
- 기본 4개: 층 두께, 레이저 파워, 스캔 속도, 에너지 밀도
- 파생: P/V, P·t, V·t, P/(V·t) 및 그 log1p 변형, Polynomial(2차) 변형
- target: 비커스 (HV)

### 가장 성능이 좋았던 모델
- **{br['model']}** on **{br['feature_set']}**
- best params: {br['best_params']}

### R² / MAE 결과 요약
- Train R²       = {br['train_R2']:+.4f}
- Test  R²       = {br['test_R2']:+.4f}
- Test  MAE      = {br['test_MAE']:.3f} HV
- Test  RMSE     = {br['test_RMSE']:.3f} HV
- 5-Fold CV R²   = {br['CV5_R2_mean']:+.4f} ± {br['CV5_R2_std']:.4f}
- 5-Fold CV MAE  = {br['CV5_MAE_mean']:.3f} HV
- LOOCV R²       = {br['LOOCV_R2']:+.4f}
- LOOCV MAE      = {br['LOOCV_MAE']:.3f} HV

### 성능이 낮거나 불안정한 경우 가능한 원인
1) 표본 수 부족 (n≈{len(y)}) — 5-Fold/LOOCV 변동 큼
2) 에너지 밀도와 P/(V·t) 의 다중공선성 → 선형 모델 불안정
3) hatch spacing 등 미관측 공정변수 — 동일 (P,V,t,E) 위 다중 측정이면 모델이 잡음으로 인식
4) 후열처리 여부, 측정 하중, 장비 변종 등 혼재 요인 미통제
5) 비커스 값 자체의 측정 분산 (반복측정 평균이 아닐 수 있음)

### 향후 개선 방향
1) 외부 오픈데이터 추가 — 논문처럼 literature-mined 데이터를 합쳐 표본 수 확대
2) porosity / OM image feature 추가 — `이미지_OM_단면/` 의 단면 이미지에서 CNN 피처 또는 정량 결함지표 추출
3) 열처리 여부 / 측정 조건 통일 — 전·후열처리 분리 모델, Vickers 하중 통일
4) 데이터 수 증가 — 신규 시편/측정 추가, 동일 조건 반복측정으로 노이즈 추정
5) SHAP / Permutation Importance 기반 해석 — 발표용 변수 기여도 정량화

### 표현 원칙
본 결과는 "논문 재현"이 아니라 "논문 방법론을 본 DB 에 적용한 **예비 모델**" 이다.
절대 R² 가 낮더라도 모델 패밀리/피처셋 간 상대비교는 의미가 있다.
""")

## 11. 산출물

- `results/paper_method_full_comparison.csv` — 모든 (모델, 피처셋) 조합의 train/test/CV5/LOOCV 지표
- `results/paper_method_R2_MAE.png` — 모델×피처셋 LOOCV R² / MAE 막대 비교
- `results/paper_method_best_actual_vs_pred.png` — Best 조합의 Actual vs Predicted + Residual
- `results/paper_method_feature_importance.png` — feature_importances_ + permutation importance